**Fig. 4D and Video S4: whole-tissue run on the placode scale stage with the DSM pair**

Needs a GPU.

In [ ]:
# Working directory: data/fig4/ under the repository root, where the Fig. 4 deposit folder is downloaded
# (domain-specialized_finetuned_bootstrap/ = the DSM model pair, one folder per sample with its
# input volume and the final mask, wound_finetuning/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig4")
print("working directory:", os.getcwd())


# EpiCycle ScaledSegmentation Pipeline

Run 3D cell segmentation, reconstruction, and assembly on tiled image data.

**Usage:** Fill in the paths and parameters below, then Run All.

## 1. Paths & Input

In [ ]:
# ===== PATHS (edit these) =====

# Input Z-stack TIFF (the original microscope file name is in the README) — set this to tile from scratch, or leave None if already tiled
INPUT_TIF = "scale_placode/input.tif"   # Set to None if tiles already exist

# Working directory (tiles + outputs go here)
# If INPUT_TIF is set, tiles will be created at WORK_DIR/tiles/
# If INPUT_TIF is None, WORK_DIR should contain existing tile subdirectories
WORK_DIR = "scale_placode"

# CellPose model paths (.pth files)
LATERAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True"
AXIAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootrap4Finetuned_axial_500epoch_scratch_lenient_rescale_True"

# Output directory name (created inside WORK_DIR)
OUTPUT_NAME = "Placode"

## 2. Resource Configuration

In [ ]:
# ===== RESOURCES (auto-detected from the job's allocation if None) =====

N_GPUS = None            # None = auto-detect from SLURM/CUDA
N_CPU_PER_WORKER = None  # None = auto (total_cores / n_gpus in parallel mode)

## 3. Stage Selection & Tiling Parameters

In [ ]:
# Which stages to run (set False to skip)
RUN_TILING = INPUT_TIF is not None   # Auto: tile if input TIFF provided
# RUN_TILING = False
RUN_SEGMENTATION = True
RUN_RECONSTRUCTION = True
RUN_ASSEMBLY = True 
RUN_GLOBAL_RECONSTRUCTION = True

# ===== TILING (Stage 0) =====
PATCH_SIZE = 1024           # Target XY patch size in pixels
# OVERLAP is shared with assembly (set in Section 6)
OVERLAP = 0.25

Z_THRESHOLD_METHOD = "hard"  # "hard", "percentile", or "default"
Z_THRESHOLD_VALUE = 0.07     # Intensity threshold for Z-range detection
CHANNEL_INDEX = 1            # Which channel for Z-range analysis
AUTO_CORRECT_PATCH = False   # Auto-find valid patch size if exact tiling fails

## 4. Segmentation Parameters

In [ ]:
# ===== SEGMENTATION (Stage 1) =====
# Recorded for reference: the stage is built without segment_params below, so the engine
# defaults were used (channels [2, 1], cell probability 0, flow threshold 0.2). The flow
# threshold acts on the 2D masks only; the 3D reconstruction uses the probability and flow maps.

# GPU allocation strategy: "auto", "fixed" (3-GPU triplets), "round_robin"
GPU_STRATEGY = "auto"

# CellPose parameters
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0
CHANNELS = [2, 1]      # [signal, nucleus] — adjust for your data
BATCH_SIZE = 1024

## 5. Reconstruction Parameters

In [ ]:
# ===== RECONSTRUCTION (Stage 2) =====

# Per-tile direct reconstruction: the engine's u-Segment3D defaults, with gradient renormalisation
# off as in the engine version the runs used (EpiCycle v0.1.2 defaults to on).
RECON_RENORM_GRADIENT = False

# "sequential" (one tile at a time, all cores) or "parallel" (multiple tiles)
RECON_MODE = "parallel"
# RECON_MODE = "sequential"

# Max concurrent tiles in parallel mode (None = n_gpus)
# Rule of thumb: 1900 TB node / ~450 GB per 1920x1920 tile ≈ 4 max
RECON_MAX_WORKERS = None

# Post-processing
SAVE_PROB3D = True
SAVE_FLOW3D = True
SAVE_BINARY = True
BINARY_ERODE = True
BINARY_KERNEL_SIZE = 3
BINARY_BORDER_MARGIN = 2
BINARY_FILL_HOLES = True

# Delete stage1 intermediates after reconstruction to save disk
CLEANUP_INTERMEDIATES = False

## 6. Assembly Parameters

In [ ]:
# ===== ASSEMBLY (Stage 3) =====

# Full volume dimensions in pixels (H, W)
FULL_SIZE = (1792, 1792)

# Tile size and overlap (must match how tiles were cut)
TILE_SIZE = PATCH_SIZE

# Number of Z slices (None = auto-detect from first tile)
DEPTH = None

# Which volumes to stitch
STITCH_MASK = True
STITCH_PROB = True
STITCH_BINARY = True

# Overlap blending for probabilities: "average" or "max"
PROB_STITCH_MODE = "max"

## 7. Global Indirect Reconstruction Parameters

In [ ]:
# ===== GLOBAL INDIRECT RECONSTRUCTION (Stage 4) =====

DOWNSAMPLE_FACTOR = 2          # Downsample stitched volume by this factor
GLOBAL_MIN_SIZE = 1000         # Remove objects smaller than this (voxels, after downsampling)
USE_PRECOMPUTED_BINARY = True  # Use stitched binary mask as foreground constraint

# Gradient descent parameters
# GLOBAL_N_ITER = 30            # Gradient descent iterations (more = finer, slower)
GLOBAL_N_ITER = 300            # Gradient descent iterations (more = finer, slower)
GLOBAL_MOMENTA = 0.9          # Momentum for gradient descent
# GLOBAL_GRADIENT_DECAY = 0.001    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.01    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.1    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.05    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
GLOBAL_GRADIENT_DECAY = 0.25    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower

GLOBAL_RENORM_GRADIENT = True  # Renormalize gradient at each step

# Indirect method parameters
# GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_SMOOTH_SKEL_SIGMA = 1   # Skeleton smoothing sigma
GLOBAL_N_CPU = 64              # CPU threads for multiprocess watershed
GLOBAL_TILE_SHAPE = None       # Tile shape for watershed (None = auto)

## 8. Setup & Pre-flight Check

In [ ]:
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

# Resolve GPU count
from packages.epicycle_scaledseg.utils.resources import get_allocated_gpus
n_gpus = N_GPUS or get_allocated_gpus()

work_dir = Path(WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)

# Output lives inside the working directory
output_dir = work_dir / OUTPUT_NAME
output_dir.mkdir(exist_ok=True)
log_dir = output_dir / "logs"
log_dir.mkdir(exist_ok=True)

# Tiles directory
tiles_dir = work_dir / "tiles"

if RUN_TILING:
    print(f"Input TIFF: {INPUT_TIF}")
    print(f"Tiles will be created in: {tiles_dir}")
    print(f"Patch size: {PATCH_SIZE}, overlap: {OVERLAP}")
else:
    # Discover existing tile directories
    tile_dirs = sorted([
        str(p) for p in work_dir.iterdir()
        if p.is_dir() and not p.name.startswith(("stage1", "stage2", "stage3", "stage4", "logs", ".", "epicycle"))
        and p.name != OUTPUT_NAME
    ])
    print(f"Found {len(tile_dirs)} existing tile directories in {work_dir}")
    for td in tile_dirs[:5]:
        print(f"  {Path(td).name}")
    if len(tile_dirs) > 5:
        print(f"  ... and {len(tile_dirs) - 5} more")

print(f"\nGPUs: {n_gpus}")
print(f"Output: {output_dir}")

Input TIFF: <run>/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL8.5_placode_003 - Stitched - Denoised.nd2-PROCESSED-ISO-1792x1792.tif
Tiles will be created in: <run>/WT_Developing_Fish/Placode/tiles
Patch size: 1024, overlap: 0.25

GPUs: 3
Output: <run>/WT_Developing_Fish/Placode/Placode


## 9. Build & Run Pipeline

In [ ]:
from packages.epicycle_scaledseg.core.pipeline import EpiCyclePipeline

pipeline = EpiCyclePipeline(
    work_dir=str(work_dir),
    n_gpus=n_gpus,
    log_dir=str(log_dir),
)

# --- Stage 0: Tiling ---
if RUN_TILING:
    from packages.epicycle_scaledseg.core.tiling import TilingStage
    pipeline.add_stage(TilingStage(
        input_tif=INPUT_TIF,
        output_dir=str(tiles_dir),
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        z_threshold_method=Z_THRESHOLD_METHOD,
        threshold_value=Z_THRESHOLD_VALUE,
        channel_index=CHANNEL_INDEX,
        auto_correct=AUTO_CORRECT_PATCH,
    ))
    # Tiling creates tile_dirs; use placeholder for now
    tile_dirs = [str(work_dir)]

# --- Stage 1: Segmentation ---
if RUN_SEGMENTATION:
    from packages.epicycle_scaledseg.core.segmentation import SegmentationStage
    pipeline.add_stage(SegmentationStage(
        lateral_model=LATERAL_MODEL,
        axial_model=AXIAL_MODEL,
        out_root=str(output_dir / "stage1_segmentation"),
        gpu_strategy=GPU_STRATEGY,
    ))

# --- Stage 2: Reconstruction ---
if RUN_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.reconstruction import ReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_reconstruct_params
    reconstruct_params = get_default_reconstruct_params()
    reconstruct_params['gradient_descent']['renorm_gradient'] = RECON_RENORM_GRADIENT
    pipeline.add_stage(ReconstructionStage(
        seg_root=str(output_dir / "stage1_segmentation"),
        out_root=str(output_dir / "stage2_reconstruction"),
        mode=RECON_MODE,
        reconstruct_params=reconstruct_params,
        max_workers=RECON_MAX_WORKERS,
        n_cpu_per_worker=N_CPU_PER_WORKER,
        save_prob3d=SAVE_PROB3D,
        save_flow3d=SAVE_FLOW3D,
        save_binary=SAVE_BINARY,
        binary_erode=BINARY_ERODE,
        binary_kernel_size=BINARY_KERNEL_SIZE,
        binary_border_margin=BINARY_BORDER_MARGIN,
        binary_fill_holes=BINARY_FILL_HOLES,
        cleanup_intermediates=CLEANUP_INTERMEDIATES,
    ))

# --- Stage 3: Assembly ---
if RUN_ASSEMBLY:
    from packages.epicycle_scaledseg.core.assembly import AssemblyStage
    pipeline.add_stage(AssemblyStage(
        recon_root=str(output_dir / "stage2_reconstruction"),
        output_dir=str(output_dir / "stage3_assembly"),
        full_size=FULL_SIZE,
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
        depth=DEPTH,
        stitch_mask=STITCH_MASK,
        stitch_prob=STITCH_PROB and SAVE_PROB3D,
        stitch_binary=STITCH_BINARY and SAVE_BINARY,
        prob_stitch_mode=PROB_STITCH_MODE,
    ))

# --- Stage 4: Global Indirect Reconstruction ---
if RUN_GLOBAL_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.global_reconstruction import GlobalReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_indirect_reconstruct_params

    indirect_params = get_default_indirect_reconstruct_params(n_cpu=GLOBAL_N_CPU)
    indirect_params['gradient_descent']['n_iter'] = GLOBAL_N_ITER
    indirect_params['gradient_descent']['momenta'] = GLOBAL_MOMENTA
    indirect_params['gradient_descent']['gradient_decay'] = GLOBAL_GRADIENT_DECAY
    indirect_params['gradient_descent']['renorm_gradient'] = GLOBAL_RENORM_GRADIENT
    indirect_params['indirect_method']['dtform_method'] = GLOBAL_DTFORM_METHOD
    indirect_params['indirect_method']['smooth_skel_sigma'] = GLOBAL_SMOOTH_SKEL_SIGMA
    if GLOBAL_TILE_SHAPE is not None:
        indirect_params['gradient_descent']['tile_shape'] = GLOBAL_TILE_SHAPE

    pipeline.add_stage(GlobalReconstructionStage(
        assembly_dir=str(output_dir / "stage3_assembly"),
        output_dir=str(output_dir / "stage4_global"),
        downsample_factor=DOWNSAMPLE_FACTOR,
        min_size=GLOBAL_MIN_SIZE,
        use_precomputed_binary=USE_PRECOMPUTED_BINARY,
        indirect_params=indirect_params,
        n_cpu=GLOBAL_N_CPU,
        tile_shape=GLOBAL_TILE_SHAPE,
    ))

print(f"Pipeline: {len(pipeline.stages)} stages")
for s in pipeline.stages:
    print(f"  - {s.name}")
print(f"Tile dirs: {len(tile_dirs)}")
print(f"Output: {output_dir}")

# ===== RUN =====
result = pipeline.run(tile_dirs)
print()
print(result.summary())

2026-02-04 13:12:25,956 epicycle.pipeline INFO Pipeline starting: 5 stage(s), 1 tile(s), 3 GPU(s)
2026-02-04 13:12:25,957 epicycle.pipeline INFO Run parameters saved to data/WT_Developing_Fish/Placode/Placode/logs/run_params.json
2026-02-04 13:12:25,957 epicycle.pipeline INFO === Stage: tiling ===
2026-02-04 13:12:25,957 epicycle.tiling INFO Stage 'tiling' starting with 1 tile(s)
2026-02-04 13:12:25,958 epicycle.tiling INFO Parameters: {"input_tif": "data/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL8.5_placode_003 - Stitched - Denoised.nd2-PROCESSED-ISO-1792x1792.tif", "patch_size": 1024, "overlap": 0.25, "z_threshold_method": "hard", "threshold_value": 0.07, "channel_index": 1, "auto_correct": false}
2026-02-04 13:12:25,958 epicycle.core.tiling INFO Tiling data/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL8.5_placode_003 - Stitched - Denoised.nd2-PROCESSED-ISO-1792x1792.tif -> data/WT_Developing_Fish/Placode/tiles (patch=1024, overlap=0.25)
2026-02-04 13

Pipeline: 5 stages
  - tiling
  - segmentation
  - reconstruction
  - assembly
  - global_reconstruction
Tile dirs: 1
Output: <run>/WT_Developing_Fish/Placode/Placode


2026-02-04 13:12:29,885 epicycle.utils.tiling INFO Z-stack shape: Z=344, C=2, H=1792, W=1792, dtype=uint16
2026-02-04 13:12:29,886 epicycle.utils.tiling INFO Tiling: patch_size=1024, step=768, overlap=0.25
2026-02-04 13:12:29,886 epicycle.utils.tiling INFO Grid: 2 rows x 2 cols = 4 tiles
Saving tiles: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.39tile/s]
2026-02-04 13:12:32,775 epicycle.core.tiling INFO Tiling complete: 4 tiles, actual patch=1024, step=768
2026-02-04 13:12:32,783 epicycle.pipeline INFO Stage 'tiling': 4/4 succeeded (6.8s)
2026-02-04 13:12:32,784 epicycle.tiling INFO Stage 'tiling' finished: 4/4 succeeded (6.8s)
2026-02-04 13:12:32,784 epicycle.pipeline INFO Stage 'tiling' produced 4 tile directories
2026-02-04 13:12:32,784 epicycle.pipeline INFO === Stage: segmentation ===
2026-02-04 13:12:32,785 epicycle.segmentation INFO Stage 'segmentation' starting with 4 tile(s)
2026

Automatric binary thresholding:  0.6
[ 292 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  108
start dynamics
Tile id 11 0 0
Tile id 5 0 0
Tile id 3 120 120
Tile id 10 0 0
Tile id 15 0 0
Tile id 9 13 13
Tile id 4 0 0
Tile id 16 331 331
Tile id 8 2259 2259
Tile id 14 8082 8082
Tile id 13 12234 12234
Tile id 18 14236 14236
Tile id 12 28452 28452
Tile id 20 43447 43447
Tile id 17 17608 17608
Tile id 19 38239 38239
Tile id 2 70457 70457
Tile id 0 72823 72823
Tile id 6 137230 137230
Tile id 7 137667 137667
Tile id 72 33156 33156
Tile id 1 236876 236876
Tile id 24 230847 230847
Tile id 21 302659 302659
Tile id 73 103121 103121
Tile id 30 326404 326404
Tile id 25 391072 391072
Tile id 71 268758 268758
Tile id 74 299640 299640
Tile id 78 300295 300295
Tile id 31 593237 593237
Tile id 23 603813 603813
Tile id 77 407826 407826
Tile id 22 683252 683252
Tile id 41 697781 697781
Tile id 84 315033 315033
Tile id 26 787229 787229
Tile id 35 830904 830904
Tile id 75 596568 596

Eroding (GPU): 100%|██████████| 964/964 [00:00<00:00, 4741.99inst/s]
GPU erosion failed (CUDA out of memory. Tried to allocate 15.86 GiB. GPU 0 has a total capacty of 139.80 GiB of which 11.24 GiB is free. Including non-PyTorch memory, this process has 128.56 GiB memory in use. Of the allocated memory 127.88 GiB is allocated by PyTorch, and 3.43 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF), falling back to CPU
Eroding (CPU): 100%|██████████| 964/964 [00:01<00:00, 713.37inst/s] 


Automatric binary thresholding:  0.5
[ 366 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  144
start dynamics
Tile id 32 0 0
Tile id 35 0 0
Tile id 29 0 0
Tile id 27 543 543
Tile id 28 962 962
Tile id 65 0 0
Tile id 31 13067 13067
Tile id 30 7094 7094
Tile id 33 7469 7469
Tile id 34 17517 17517
Tile id 26 61013 61013
Tile id 11 97203 97203
Tile id 64 8906 8906
Tile id 1 91629 91629
Tile id 70 0 0
Tile id 71 0 0
Tile id 25 115863 115863
Tile id 24 118955 118955
Tile id 0 133829 133829
Tile id 17 119213 119213
Tile id 6 135695 135695
Tile id 5 168508 168508
Tile id 12 183957 183957
Tile id 69 20281 20281
Tile id 3 217975 217975
Tile id 21 223830 223830
Tile id 7 264986 264986
Tile id 23 267474 267474
Tile id 4 343096 343096
Tile id 2 349216 349216
Tile id 15 376971 376971
Tile id 63 275023 275023
Tile id 18 386621 386621
Tile id 66 261201 261201
Tile id 9 385227 385227
Tile id 68 211332 211332
Tile id 59 388279 388279
Tile id 20 415644 415644
Tile id 22 463855 46

... (4 outputs left out)

Eroding (GPU): 100%|██████████| 1057/1057 [00:00<00:00, 5429.56inst/s]
2026-02-04 13:30:36,837 epicycle.pipeline INFO Stage 'reconstruction': 4/4 succeeded (580.8s)
2026-02-04 13:30:36,838 epicycle.reconstruction INFO Stage 'reconstruction' finished: 4/4 succeeded (580.8s)
2026-02-04 13:30:36,838 epicycle.pipeline INFO === Stage: assembly ===
2026-02-04 13:30:36,839 epicycle.assembly INFO Stage 'assembly' starting with 4 tile(s)
2026-02-04 13:30:36,839 epicycle.assembly INFO Parameters: {"full_size": [1792, 1792], "tile_size": 1024, "overlap": 0.25, "depth": null, "stitch_mask": true, "stitch_prob": true, "stitch_binary": true, "prob_stitch_mode": "max", "binary_format": "npy", "n_workers": 16}
2026-02-04 13:30:36,840 epicycle.assembly INFO Loading tiling config from data/WT_Developing_Fish/Placode/tiles/tiling_config.json
2026-02-04 13:30:36,840 epicycle.assembly INFO Loading tiling config from data/WT_Developing_Fish/Placode/tiles/tiling_config.json
2026-02-04 13:30:36,841 epicycle.a

[ 268 1088 1088] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  108
start dynamics
Tile id 30 646 646
Tile id 19 333 333
Tile id 24 646 646
Tile id 35 11697 11697
Tile id 31 11166 11166
Tile id 20 5880 5880
Tile id 34 44073 44073
Tile id 29 47305 47305
Tile id 23 43570 43570
Tile id 18 12253 12253
Tile id 33 58738 58738
Tile id 25 19105 19105
Tile id 14 18379 18379
Tile id 41 61920 61920
Tile id 0 36361 36361
Tile id 5 83009 83009
Tile id 13 49889 49889
Tile id 17 88669 88669
Tile id 32 85947 85947
Tile id 77 0 0
Tile id 72 0 0
Tile id 73 0 0
Tile id 75 0 0
Tile id 74 0 0
Tile id 76 0 0
Tile id 12 90615 90615
Tile id 83 0 0
Tile id 59 126953 126953
Tile id 78 0 0
Tile id 11 137200 137200
Tile id 1 113010 113010
Tile id 53 149803 149803
Tile id 82 0 0
Tile id 89 0 0
Tile id 15 116363 116363
Tile id 47 157849 157849
Tile id 95 0 0
Tile id 6 163435 163435
Tile id 94 0 0
Tile id 96 142 142
Tile id 21 178817 178817
Tile id 71 15241 15241
Tile id 65 90300 90300
Tile id 101 0 0

2026-02-04 13:34:41,872 epicycle.core.global_reconstruction INFO Global reconstruction complete: shape=(172, 896, 896), n_labels=2432, elapsed=211.0s
2026-02-04 13:34:41,873 epicycle.core.global_reconstruction INFO Saving outputs to <run>/WT_Developing_Fish/Placode/Placode/stage4_global
2026-02-04 13:34:54,429 epicycle.pipeline INFO Stage 'global_reconstruction': 1/1 succeeded (229.1s)
2026-02-04 13:34:54,429 epicycle.global_reconstruction INFO Stage 'global_reconstruction' finished: 1/1 succeeded (229.1s)
2026-02-04 13:34:54,430 epicycle.pipeline WARNING No tiles remaining — stopping pipeline.
2026-02-04 13:34:54,430 epicycle.pipeline INFO Pipeline: 1/4 tiles (1348.5s total)
  tiling: 4/4 succeeded (6.8s)
  segmentation: 4/4 succeeded (503.3s)
  reconstruction: 4/4 succeeded (580.8s)
  assembly: 4/4 succeeded (28.5s)
  global_reconstruction: 1/1 succeeded (229.1s)



Pipeline: 1/4 tiles (1348.5s total)
  tiling: 4/4 succeeded (6.8s)
  segmentation: 4/4 succeeded (503.3s)
  reconstruction: 4/4 succeeded (580.8s)
  assembly: 4/4 succeeded (28.5s)
  global_reconstruction: 1/1 succeeded (229.1s)


## 10. Check Status

In [ ]:
# Quick status check — how many tiles completed each stage?
seg_root = output_dir / "stage1_segmentation"
recon_root = output_dir / "stage2_reconstruction"

seg_done = sum(1 for td in tile_dirs if (seg_root / Path(td).name / "SEGMENTATION_DONE").exists())
recon_done = sum(1 for td in tile_dirs if (recon_root / Path(td).name / f"{Path(td).name}_mask.tif").exists())

print(f"Stage 1 (Segmentation):   {seg_done}/{len(tile_dirs)} complete")
print(f"Stage 2 (Reconstruction): {recon_done}/{len(tile_dirs)} complete")
print(f"Stage 3 (Assembly):       check {output_dir / 'stage3_assembly'}")
print(f"Stage 4 (Global Recon):   {'done' if (output_dir / 'stage4_global' / 'global_segmentation.tif').exists() else 'not done'}")

Stage 1 (Segmentation):   0/1 complete
Stage 2 (Reconstruction): 0/1 complete
Stage 3 (Assembly):       check <run>/WT_Developing_Fish/Placode/Placode/stage3_assembly
Stage 4 (Global Recon):   done
